Custom feature functions

In [6]:
import pandas as pd
import numpy as np
import spacy
import itertools as it
nlp = spacy.load('en_core_web_sm')
import re

In [7]:
# Entity labels in spacy
nlp.get_pipe('ner').labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

#### Entity labels

In [8]:
entity_labels = ['', 'A', 'E', 'O']
entity_iob = ['B', 'I', 'O']

#### Feature functions

In [9]:
#1 Starts with upper case
def is_upper(doc):
    upper = re.compile('Xx+')
    condition = lambda x: 1 if upper.match(x) else 0
    return np.array([condition(tok.shape_) for tok in nlp(doc)])


In [10]:
#2 Noun following a noun
def is_noun_noun(doc):
    pos = [tok.pos_ for tok in nlp(doc)]
    condition = lambda i: 1 if pos[i] in ('NOUN', 'PROPN') and pos[i-1] in ('NOUN', 'PROPN') else 0
    return np.array([0] + [condition(i) for i in range(1, len(pos))])


In [11]:
#3 Noun following a verb
def is_noun_verb(doc):
    pos = [tok.pos_ for tok in nlp(doc)]
    condition = lambda i: 1 if pos[i] in ('NOUN', 'PROPN') and pos[i-1] in ('VERB') else 0
    return np.array([0] + [condition(i) for i in range(1, len(pos))])


In [12]:
#4 Entity E following a verb
def is_E_verb(doc, labels):
    pos = [tok.pos_ for tok in nlp(doc)]
    condition = lambda i: 1 if pos[i-1] == 'VERB' and labels[i] != 'E' else 0
    return np.array([0]+[condition(i) for i in range(1, len(pos))])


In [13]:
#5 Preposition following a preposition
def is_prep_prep(doc):
    pos = [tok.pos_ for tok in nlp(doc)]
    condition = lambda i: 1 if pos[i-1] == 'ADP' and pos[i] == 'ADP' else 0
    return np.array([0]+[condition(i) for i in range(1, len(pos))])

In [42]:
#6 INC following ORG
def is_Org_Inc(doc, labels):
    text = [tok.text for tok in nlp(doc)]
    condition = lambda i: 1 if text[i].lower() == 'inc' and labels[i-1] == 'ORG' else 0
    return np.array([0]+[condition(i) for i in range(1, len(text))])

In [15]:
#7 Last word LOC
def is_last_LOC(labels):
    condition = lambda i: 1 if i==len(labels)-1 and labels[i]=='LOC' else 0
    return [condition(i) for i in range(len(labels))]

In [ ]:
#8 Prof/Dr then PERSON
def is_pref_person(doc, labels):
    text = [tok.text for tok in nlp(doc)]
    condition = lambda i: 1 if labels[i] == 'B-PERSON' and text[i-1].lower() in ('prof', 'dr') else 0
    return [condition(i) for i in range(len(text))]

In [69]:
sent = 'Dr Tony is teaching Artificial Intelligence in Australia'
labels = ['I-PREFIX', 'I-PERSON', 'O', 'O', 'B-LOCATION', 'I-LOCATION', 'O', 'B-LOCATION']

In [70]:
def feature_array(doc, labels):
    f_array = np.c_[np.array([is_upper(doc),  
                              is_pref_person(doc, labels),
                              is_noun_noun(doc)
                              ])]
    return f_array

In [67]:
print(len(nlp(sent)))

8


How to make all possible permutations of entity labels with repititions for given number of words in sequence

In [71]:
W = np.array([list(range(1, 4))])
X = feature_array(sent, labels)
X

array([[1, 1, 0, 0, 1, 1, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 1, 0, 0]])

In [72]:
W.dot(X).sum()

np.int64(11)